In [7]:
#we begin by keeping lmbda = 0 , i.e. no penalty. this is done to see natural behaviour of points in Poincare space
#under difference base prior scales - sigma_u

%load_ext autoreload
%autoreload 2

import sys
import os

# Start at the notebook's current folder
current_dir = os.getcwd()

# Climb up directories until we find the folder containing 'src'
while current_dir != os.path.dirname(current_dir):  # Stop if we hit the file system root
    if "src" in os.listdir(current_dir):
        break
    current_dir = os.path.dirname(current_dir)

# Append the absolute project root to Python's search path
if current_dir not in sys.path:
    sys.path.append(current_dir)

print(f"Project root successfully found and added: {current_dir}")

import torch
import pyro
from pyro.infer import MCMC, NUTS
import pandas as pd

from src.model import PhylogeneticPrior
from src.diagnostics import get_fresh_test_quartets, evaluate_test_diagnostics


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Project root successfully found and added: /Users/riyaaggarwal/Desktop/Block4/Block4 project/scripts


In [8]:
N = 50
K = 2
B_train = 200  # Stays fixed, but penalty is turned off via lambda=0
B_test = 1000


print("Initializing invariant test dataset...")
baseline_model = PhylogeneticPrior(N=N, K=K, B=B_train, seed=42)
# Generate fixed validation quartets
test_quartets = get_fresh_test_quartets(
    N=N, B_test=B_test, train_quartets=baseline_model.fixed_indices, seed=123
)

Initializing invariant test dataset...


In [9]:
# Define the grid sweep for sigma_u
sigma_u_grid = [0.1, 0.5, 1.0, 2.0, 5.0]

# Dictionary to collect results for comparison
experiment_results = []

for sigma_u in sigma_u_grid:
    print("\n" + "="*60)
    print(f" RUNNING EXPERIMENT 1: sigma_u = {sigma_u} (lambda = 0.0)")
    print("="*60)
    
    # 1. Reset Pyro's parameter store and fix random seed for the sampler
    pyro.clear_param_store() #clears global bg memory so parameters for 0.1 sigma_u do not leak into run for 0.5
    pyro.set_rng_seed(44)
    
    # 2. Instantiate model with identical fixed training setup
    model = PhylogeneticPrior(N=N, K=K, B=B_train, seed=42)
    
    # 3. Configure NUTS kernel on your generative function
    nuts_kernel = NUTS(model.initialize)
    mcmc = MCMC(
        nuts_kernel, 
        num_samples=400,    # Good size for swift prior diagnostics
        warmup_steps=200, 
        num_chains=1
    )
    
    # 4. Execute the chain
    mcmc.run(lmbda=0.0, sigma_u=sigma_u, tau=0.1) #lmbda = 0 skips the tree penalty calculations entirely. 
    #Just let NUTS wander freely through the raw unconstrained space.

    # 5. Extract posterior samples for latent variables (like 'u')
    posterior_samples = mcmc.get_samples()
    
    # --- FIX: Use Predictive to explicitly regenerate and capture 'D' ---
    from pyro.infer import Predictive
    
    # This runs a forward pass through your model using the sampled 'u' coordinates
    predictive = Predictive(model.initialize, posterior_samples, return_sites=["D"])
    # Run the predictive trace (we pass dummy values for the parameters)
    predictive_samples = predictive(lmbda=0.0, sigma_u=sigma_u, tau=0.1)
    
    # This guarantees you get your 'D_samples' tensor of shape (S, N, N)!
    D_samples = predictive_samples["D"].squeeze(1) if predictive_samples["D"].dim() == 4 else predictive_samples["D"]
    
    # 6. Evaluate metrics using your vectorized global diagnostic compiler
    metrics = evaluate_test_diagnostics(D_samples, test_quartets, epsilon=0.05, gamma=0.05)
    
    # 7. Collect diagnostics alongside NUTS trace stats
    sampler_diagnostics = mcmc.diagnostics()
    # Pyro stores step-by-step divergence flags as a list of bools/integers under 'divergent'
    divergence_list = sampler_diagnostics.get("divergent", [0])
    num_divergences = int(sum(divergence_list))
    
    summary_row = {
        "sigma_u": sigma_u,
        "Min_Dist": round(metrics["distance_scale"]["min"], 4),
        "Median_Dist": round(metrics["distance_scale"]["median"], 4),
        "Max_Dist": round(metrics["distance_scale"]["max"], 4),
        "Divergences": num_divergences,
        "Test_Violations_Median": round(metrics["hard_violations"]["median"], 4)
    }
    experiment_results.append(summary_row)



    # --- Final Analysis Table Output ---
print("\n" + "#"*60)
print("             EXPERIMENT 1 FINAL RESULTS TABLE             ")
print("#"*60)
df = pd.DataFrame(experiment_results)
print(df.to_string(index=False))
print("#"*60 + "\n")


 RUNNING EXPERIMENT 1: sigma_u = 0.1 (lambda = 0.0)


Sample: 100%|██████████| 600/600 [00:05, 112.97it/s, step size=4.45e-01, acc. prob=0.852]



 RUNNING EXPERIMENT 1: sigma_u = 0.5 (lambda = 0.0)


Sample: 100%|██████████| 600/600 [00:05, 114.07it/s, step size=4.09e-01, acc. prob=0.876]



 RUNNING EXPERIMENT 1: sigma_u = 1.0 (lambda = 0.0)


...


 RUNNING EXPERIMENT 1: sigma_u = 2.0 (lambda = 0.0)


Sample: 100%|██████████| 600/600 [00:05, 101.64it/s, step size=4.28e-01, acc. prob=0.875]



 RUNNING EXPERIMENT 1: sigma_u = 5.0 (lambda = 0.0)


Sample: 100%|██████████| 600/600 [00:04, 123.37it/s, step size=4.57e-01, acc. prob=0.853]



############################################################
             EXPERIMENT 1 FINAL RESULTS TABLE             
############################################################
 sigma_u  Min_Dist  Median_Dist  Max_Dist  Divergences  Test_Violations_Median
     0.1    0.0007       0.3332    1.4543            0                  0.0488
     0.5    0.0043       1.7786    7.6401            0                  0.2139
     1.0    0.0064       3.8822   14.3268            0                  0.3328
     2.0    0.0021       8.6326   18.8906            0                  0.4449
     5.0    0.0639      17.5556   18.8907            0                  0.9417
############################################################

